<a href="https://colab.research.google.com/github/vika3mm/gost_progect/blob/main/%D1%81%D0%B1%D0%BE%D1%80%D0%BA%D0%B0_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers pymupdf python-docx pillow opencv-python numpy

import os, re, fitz, torch, numpy as np
from PIL import Image
from collections import Counter
from google.colab import drive
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.enum.text import WD_TAB_ALIGNMENT, WD_TAB_LEADER
from docx.enum.style import WD_STYLE_TYPE
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
drive.mount('/content/drive')

MODEL_PATH = "/content/drive/MyDrive/GOST_Project/layoutlm_model_split"
INPUT_PDF = "/content/drive/MyDrive/GOST_Project/Денисова_ВКР_поломанный.pdf"
OUTPUT_DOCX = "/content/drive/MyDrive/GOST_Project/smart_result_new2.docx"

processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)
model = LayoutLMv3ForTokenClassification.from_pretrained(MODEL_PATH)
model.eval()

id2label = {0: 'BODY', 1: 'FIGURE', 2: 'FIGURE_CAPTION', 3: 'FOOTNOTE', 4: 'FORMULA',
            5: 'OTHER', 6: 'REFERENCE', 7: 'SECTION_HEADER', 8: 'TABLE',
            9: 'TABLE_CAPTION', 10: 'TITLE', 11: 'TOC'}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.2 MB/s eta 0:00:00
Mounted at /content/drive


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/216 [00:00<?, ?it/s]

In [2]:
def extract_pymu_data(pdf_path):
    """
    Извлекает слова из PDF.

    Для каждого слова сохраняются:
    - text
    - bbox в системе 0..1000
    - block_no
    - line_no
    - word_no
    - word_index

    line_no и block_no потом используются для точного
    восстановления физических строк PDF.
    """
    doc = fitz.open(pdf_path)
    all_pages = {}

    for page_index in range(len(doc)):
        page = doc[page_index]

        words_data = page.get_text("words", sort=True)

        page_width = page.rect.width
        page_height = page.rect.height

        words = []

        for word_index, w in enumerate(words_data):
            x0, y0, x1, y1 = w[0], w[1], w[2], w[3]
            text = str(w[4])

            block_no = int(w[5])
            line_no = int(w[6])
            word_no = int(w[7])

            box = [
                max(0, min(1000, int(x0 / page_width * 1000))),
                max(0, min(1000, int(y0 / page_height * 1000))),
                max(0, min(1000, int(x1 / page_width * 1000))),
                max(0, min(1000, int(y1 / page_height * 1000))),
            ]

            words.append({
                "text": text,
                "box": box,

                "block_no": block_no,
                "line_no": line_no,
                "word_no": word_no,

                "word_index": word_index,
            })

        all_pages[page_index + 1] = {
            "words": words,
            "width": page_width,
            "height": page_height,
        }

    doc.close()

    return all_pages

print("📄 Извлекаем текст из PDF через PyMuPDF...")
pymu_data = extract_pymu_data(INPUT_PDF)
print(f"   Извлечено {len(pymu_data)} страниц")


def predict_page(word_items, max_words=50):
    """
    Классифицирует страницу чанками и сохраняет
    исходную структуру PyMuPDF.
    """

    if not word_items:
        return []

    valid_items = [
        item for item in word_items
        if item["text"].strip()
    ]

    all_predictions = []

    try:
        device = next(model.parameters()).device
    except Exception:
        device = torch.device("cpu")

    for start in range(0, len(valid_items), max_words):

        chunk = valid_items[start:start + max_words]

        chunk_words = [
            item["text"]
            for item in chunk
        ]

        chunk_boxes = [
            item["box"]
            for item in chunk
        ]

        image = Image.new(
            "RGB",
            (1000, 1000),
            color="white"
        )

        encoding = processor(
            image,
            chunk_words,
            boxes=chunk_boxes,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )

        # Получаем соответствие:
        # token -> исходное слово
        word_ids = encoding.word_ids(
            batch_index=0
        )

        model_inputs = {
            key: value.to(device)
            for key, value in encoding.items()
        }

        with torch.no_grad():
            outputs = model(**model_inputs)

        preds = outputs.logits.argmax(
            dim=-1
        )[0].cpu()

        used_word_ids = set()

        for token_index, word_id in enumerate(word_ids):

            if word_id is None:
                continue

            if word_id in used_word_ids:
                continue

            if word_id >= len(chunk):
                continue

            used_word_ids.add(word_id)

            original_item = chunk[word_id]

            label_id = int(
                preds[token_index].item()
            )

            all_predictions.append({
                **original_item,
                "label": id2label[label_id]
            })

    all_predictions.sort(
        key=lambda x: x["word_index"]
    )

    return all_predictions

def build_lines(predictions):
    """
    Собирает визуальные строки по геометрии bbox.

    Не доверяем block_no/line_no как истинным
    строкам документа, потому что PDF может дробить
    одну визуальную строку на несколько внутренних spans.
    """

    if not predictions:
        return []

    items = []

    for item in predictions:
        text = item["text"].strip()

        if not text:
            continue

        x0, y0, x1, y1 = item["box"]

        item = dict(item)

        item["_cy"] = (y0 + y1) / 2
        item["_height"] = max(1, y1 - y0)

        items.append(item)

    if not items:
        return []

    items.sort(
        key=lambda x: (
            x["_cy"],
            x["box"][0]
        )
    )

    visual_lines = []

    for item in items:
        x0, y0, x1, y1 = item["box"]

        cy = item["_cy"]
        height = item["_height"]

        best_line = None
        best_distance = None

        for line in reversed(visual_lines[-10:]):

            line_y0 = line["y0"]
            line_y1 = line["y1"]
            line_cy = line["cy"]

            line_height = max(
                1,
                line_y1 - line_y0
            )

            overlap = max(
                0,
                min(y1, line_y1)
                - max(y0, line_y0)
            )

            overlap_ratio = overlap / max(
                1,
                min(height, line_height)
            )

            center_distance = abs(
                cy - line_cy
            )

            same_visual_line = (
                overlap_ratio >= 0.35
                or
                center_distance <= max(
                    4,
                    min(height, line_height) * 0.45
                )
            )

            if not same_visual_line:
                continue

            if (
                best_distance is None
                or center_distance < best_distance
            ):
                best_line = line
                best_distance = center_distance

        if best_line is None:

            visual_lines.append({
                "items": [item],
                "y0": y0,
                "y1": y1,
                "cy": cy,
            })

        else:

            best_line["items"].append(item)

            best_line["y0"] = min(
                best_line["y0"],
                y0
            )

            best_line["y1"] = max(
                best_line["y1"],
                y1
            )

            best_line["cy"] = sum(
                x["_cy"]
                for x in best_line["items"]
            ) / len(best_line["items"])

    result = []

    for visual_line in visual_lines:

        line_items = sorted(
            visual_line["items"],
            key=lambda x: x["box"][0]
        )

        text = " ".join(
            x["text"]
            for x in line_items
        ).strip()

        if not text:
            continue

        labels = [
            x["label"]
            for x in line_items
        ]

        counts = Counter(labels)

        main_label = counts.most_common(1)[0][0]

        ref_count = counts.get(
            "REFERENCE",
            0
        )

        toc_count = counts.get(
            "TOC",
            0
        )

        if (
            ref_count > 0
            and
            ref_count >= len(line_items) * 0.5
        ):
            main_label = "REFERENCE"

        elif (
            toc_count > 0
            and
            toc_count >= len(line_items) * 0.5
        ):
            main_label = "TOC"

        result.append({
            "text": text,
            "label": main_label,
            "box": [
                min(x["box"][0] for x in line_items),
                min(x["box"][1] for x in line_items),
                max(x["box"][2] for x in line_items),
                max(x["box"][3] for x in line_items),
            ],
            "word_labels": labels,
        })

    result.sort(
        key=lambda x: (
            x["box"][1],
            x["box"][0]
        )
    )

    return result

print(" Классифицируем страницы.")

all_pages_lines = {}

for page_num in sorted(pymu_data.keys()):

    word_items = pymu_data[page_num]["words"]

    predictions = predict_page(
        word_items,
        max_words=50
    )

    lines = build_lines(predictions)

    for line in lines:
        line["page"] = page_num

    all_pages_lines[page_num] = lines

print(
    f"   ✅ Обработано {len(all_pages_lines)} страниц"
)

# ПОИСК СТРАНИЦ ПО ПЛОТНОСТИ МЕТОК

def find_page_by_label_density(all_pages_lines, label):
    """
    Ищет страницу, где больше всего строк нужного класса.

    Используются одновременно:
    - количество строк;
    - доля строк данного класса.
    """

    page_scores = {}

    for page_num, lines in all_pages_lines.items():

        if not lines:
            continue

        label_count = sum(
            1
            for line in lines
            if line["label"] == label
        )

        if label_count == 0:
            continue

        ratio = label_count / len(lines)

        page_scores[page_num] = {
            "count": label_count,
            "ratio": ratio,
        }

    if not page_scores:
        return None

    best_page = max(
        page_scores,
        key=lambda p: (
            page_scores[p]["count"],
            page_scores[p]["ratio"],
        )
    )

    score = page_scores[best_page]

    print(
        f"    Страница {best_page}: "
        f"{score['count']} строк {label}, "
        f"плотность {score['ratio']:.2f}"
    )

    return best_page

print("\n🔍 Ищем страницы по плотности меток...")
toc_page = find_page_by_label_density(all_pages_lines, "TOC")
ref_page = find_page_by_label_density(all_pages_lines, "REFERENCE")

print(f"    Страница содержания: {toc_page}")
print(f"    Страница библиографии: {ref_page}")

# ИЗВЛЕЧЕНИЕ СОДЕРЖАНИЯ И БИБЛИОГРАФИИ

def is_toc_heading(text):
    return text.strip().lower() in ["содержание", "оглавление"]

def is_ref_heading(text):
    return text.strip().lower() in [
        "список литературы", "библиографический список",
        "библиография", "список использованных источников"
    ]

def extract_toc(all_pages_lines, toc_page):
    """
    Извлекает содержание только из непрерывного блока
    TOC-страниц вокруг найденной основной страницы содержания.
    """

    if toc_page is None:
        return []

    page_stats = {}

    for page_num, lines in all_pages_lines.items():

        if not lines:
            continue

        toc_count = sum(
            1
            for line in lines
            if line["label"] == "TOC"
        )

        toc_ratio = (
            toc_count / len(lines)
        )

        page_stats[page_num] = {
            "toc_count": toc_count,
            "toc_ratio": toc_ratio,
            "total": len(lines),
        }


    def is_toc_page(page_num):

        if page_num not in page_stats:
            return False

        stats = page_stats[page_num]

        count = stats["toc_count"]
        ratio = stats["toc_ratio"]

        return (
            count >= 3
            or
            ratio >= 0.25
        )


    selected_pages = [toc_page]

    page = toc_page - 1

    while (
        page in all_pages_lines
        and is_toc_page(page)
    ):
        selected_pages.insert(0, page)
        page -= 1

    page = toc_page + 1

    while (
        page in all_pages_lines
        and is_toc_page(page)
    ):
        selected_pages.append(page)
        page += 1

    print(
        "   📑 Страницы содержания: "
        + ", ".join(
            map(str, selected_pages)
        )
    )

    result = []

    for page_num in selected_pages:

        for line in all_pages_lines[page_num]:

            if line["label"] != "TOC":
                continue

            text = line["text"].strip()

            if not text:
                continue

            result.append({
                **line,
                "page": page_num
            })


    result.sort(
        key=lambda x: (
            x["page"],
            x["box"][1],
            x["box"][0]
        )
    )
    if result:
        first_text = result[0]["text"].strip().lower()

        if first_text in ("содержание", "оглавление"):
            result = result[1:]
    return result


def normalize_reference_text(text):
    if text is None:
        return ""

    text = str(text)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def is_page_number_line(line):
    text = normalize_reference_text(
        line["text"]
    )

    if not re.fullmatch(r"\d{1,4}", text):
        return False

    y0 = line["box"][1]
    y1 = line["box"][3]

    return y0 <= 100 or y1 >= 900


YEAR_RE = re.compile(
    r"\("
    r"(?:"
        r"n\.d\.(?:-[a-z])?"
        r"|"
        r"(?:18|19|20)\d{2}"
        r"(?:[^)]{0,40})?"
    r")"
    r"\)",
    flags=re.IGNORECASE
)


def contains_year(text):
    return YEAR_RE.search(
        normalize_reference_text(text)
    ) is not None


def starts_with_year(text):
    text = normalize_reference_text(text)

    return re.match(
        r"^\s*(?:"
        r"\((?:18|19|20)\d{2}[^)]*\)"
        r"|"
        r"(?:18|19|20)\d{2}"
        r")"
        r"[\s.,:]",
        text
    ) is not None


def starts_with_author(text):
    """
    Smith J.
    Smith J. A.
    De Moura, L.
    """
    text = normalize_reference_text(text)

    if not text:
        return False

    if re.match(
        r"^(?:DOI|URL|ISBN|ISSN)\b",
        text,
        flags=re.IGNORECASE
    ):
        return False

    pattern1 = (
        r"^[A-ZА-ЯЁ]"
        r"[A-Za-zА-Яа-яЁё'’\-]+"
        r"(?:\s+[A-ZА-ЯЁ]\.){1,4}"
        r"(?:\s|$)"
    )

    if re.match(pattern1, text):
        return True

    pattern2 = (
        r"^[A-ZА-ЯЁ]"
        r"[A-Za-zА-Яа-яЁё'’\-]+"
        r"(?:\s+[A-Za-zА-Яа-яЁё'’\-]+){0,3}"
        r",\s*"
        r"(?:[A-ZА-ЯЁ]\.){1,4}"
    )

    return re.match(
        pattern2,
        text
    ) is not None


def looks_like_full_name_authors(text):
    """
    Только для редкой записи:
    Sang Kil Cha ... and David Brumley.

    НЕ используем функцию сама по себе.
    Только вместе с годом в следующей строке.
    """
    text = normalize_reference_text(text)

    if not re.search(
        r"\b(?:and|&)\b",
        text,
        flags=re.IGNORECASE
    ):
        return False

    bad_words = {
        "software",
        "testing",
        "security",
        "privacy",
        "analysis",
        "system",
        "systems",
        "generation",
        "techniques",
        "implications",
        "symposium",
        "conference",
        "proceedings",
        "information",
        "construction",
        "exploit",
    }

    words = re.findall(
        r"[A-Za-zА-Яа-яЁё'’\-]+",
        text
    )

    lower = {
        w.lower()
        for w in words
    }

    if lower & bad_words:
        return False

    capitalized = [
        w
        for w in words
        if w and w[0].isupper()
    ]

    return len(capitalized) >= 5


def forced_continuation(previous, current):
    previous = normalize_reference_text(previous)
    current = normalize_reference_text(current)

    # DOI / URL
    if re.match(
        r"^(?:DOI|URL|ISBN|ISSN)\s*:",
        current,
        flags=re.IGNORECASE
    ):
        return True

    if re.match(
        r"^https?://",
        current,
        flags=re.IGNORECASE
    ):
        return True

    if (
        re.search(r"\d[\-–—]$", previous)
        and
        re.match(r"^\d", current)
    ):
        return True

    if re.search(
        r"[A-Za-zА-Яа-яЁё]-$",
        previous
    ):
        return True

    if re.search(
        r"(?:&|\band)\s*$",
        previous,
        flags=re.IGNORECASE
    ):
        return True

    return False


def is_new_reference(lines, i):
    current = normalize_reference_text(
        lines[i]["text"]
    )

    if i == 0:
        return True

    previous = normalize_reference_text(
        lines[i - 1]["text"]
    )

    next_text = ""

    if i + 1 < len(lines):
        next_text = normalize_reference_text(
            lines[i + 1]["text"]
        )

    if forced_continuation(
        previous,
        current
    ):
        return False

    if re.match(
        r"^\d",
        current
    ):
        return False

    if starts_with_year(current):
        return False

    if (
        starts_with_author(current)
        and
        contains_year(current)
    ):
        return True

    if starts_with_author(current):

        if starts_with_year(next_text):
            return True

        if (
            starts_with_author(next_text)
            and
            contains_year(next_text)
        ):
            return True

        return False

    if (
        re.match(
            r"^[A-ZА-ЯЁ]",
            current
        )
        and
        contains_year(current)
    ):
        return True

    if (
        looks_like_full_name_authors(current)
        and
        starts_with_year(next_text)
    ):
        return True

    return False


def join_reference_lines(left, right):
    """
    Соединяет две физические строки одной библиографической записи.

    Не определяет границы источников — это делает is_new_reference().
    Здесь только корректно склеиваются продолжения одной записи.
    """

    left = normalize_reference_text(left)
    right = normalize_reference_text(right)

    if not left:
        return right

    if not right:
        return left

    if (
        re.search(r"\d[\-–—]$", left)
        and
        re.match(r"^\d", right)
    ):
        return left + right


    if re.search(
        r"https?://\S+$",
        left,
        flags=re.IGNORECASE
    ):
        if (
            len(right) <= 15
            and
            re.fullmatch(
                r"[A-Za-z0-9._~/%?#=&+\-]+",
                right
            )
        ):
            return left + right

    if re.search(
        r"[A-Za-zА-Яа-яЁё]-$",
        left
    ):
        return left + right

    return left + " " + right

def starts_with_explicit_reference_number(text):
    """
    Явная нумерация библиографии:
    1. ...
    2) ...
    [3] ...

    DOI вида 10.1145/... сюда не попадёт,
    потому что после точки должна идти пробельная граница.
    """
    text = normalize_reference_text(text)

    return re.match(
        r"^\s*(?:\[\d+\]|\d+[.)])\s+",
        text
    ) is not None


def get_reference_gap_thresholds(lines):
    """
    Для каждой страницы автоматически оценивает,
    какой вертикальный промежуток между строками
    уже похож не на обычный межстрочный интервал,
    а на начало нового абзаца / источника.

    Никаких значений под конкретный документ:
    порог вычисляется из геометрии самой страницы.
    """

    page_gaps = {}

    for i in range(1, len(lines)):

        prev = lines[i - 1]
        curr = lines[i]

        prev_page = prev.get("page")
        curr_page = curr.get("page")

        # Между разными страницами расстояние bbox
        # сравнивать нельзя.
        if prev_page != curr_page:
            continue

        gap = (
            curr["box"][1]
            - prev["box"][3]
        )

        # Нужны только реальные положительные промежутки.
        if gap <= 0:
            continue

        page_gaps.setdefault(
            curr_page,
            []
        ).append(gap)

    thresholds = {}

    for page_num, gaps in page_gaps.items():

        if not gaps:
            continue

        gaps = sorted(gaps)

        median_gap = float(
            np.median(gaps)
        )

        deviations = [
            abs(g - median_gap)
            for g in gaps
        ]

        mad = float(
            np.median(deviations)
        )

        # Порог deliberately выше обычного
        # межстрочного расстояния.
        #
        # Если обычный gap ~3, а между источниками ~25,
        # новый источник будет найден.
        #
        # Если все расстояния примерно одинаковы,
        # геометрия просто не будет вмешиваться,
        # и останется старый semantic parser.
        threshold = max(
            median_gap * 2.2,
            median_gap + max(
                8.0,
                mad * 4.0
            )
        )

        thresholds[
            page_num
        ] = threshold

    return thresholds


def has_strong_paragraph_gap(
    previous_line,
    current_line,
    thresholds
):
    """
    True, если между строками на одной странице
    значительно больше места, чем при обычном
    переносе строки.
    """

    prev_page = previous_line.get(
        "page"
    )

    curr_page = current_line.get(
        "page"
    )

    if prev_page != curr_page:
        return False

    if curr_page not in thresholds:
        return False

    gap = (
        current_line["box"][1]
        - previous_line["box"][3]
    )

    return (
        gap
        >= thresholds[curr_page]
    )


def is_author_list_continuation(
    previous_text,
    current_text
):
    """
    Ловит случай, когда длинный список авторов
    просто перенесён на следующую физическую строку.

    Например:

    Huang, ..., Simon, I.,
    Hawthorne, ..., Eck, D. (2018).

    Это ОДИН источник.
    """

    previous_text = normalize_reference_text(
        previous_text
    )

    current_text = normalize_reference_text(
        current_text
    )

    if not previous_text or not current_text:
        return False

    # Предыдущая строка явно оборвалась
    # посреди перечисления авторов.
    if re.search(
        r",\s*$",
        previous_text
    ):

        # Продолжение начинается фамилией
        # или инициалом.
        if re.match(
            r"^(?:"
            r"[A-ZА-ЯЁ][A-Za-zА-Яа-яЁё'’\-]+"
            r"|"
            r"[A-ZА-ЯЁ]\."
            r")",
            current_text
        ):
            return True

    # Список авторов заканчивается
    # соединительным знаком/словом.
    if re.search(
        r"(?:&|\band)\s*$",
        previous_text,
        flags=re.IGNORECASE
    ):
        return True

    return False

def group_references(lines):
    """
    Универсальное восстановление библиографических записей.

    Использует одновременно:

    1. Геометрию PDF:
       большой вертикальный промежуток между строками
       означает вероятное начало нового источника.

    2. Явную нумерацию:
       1. ...
       2) ...
       [3] ...

    3. Старый semantic parser is_new_reference().

    4. Защиту от ложного разделения длинного
       списка авторов на две записи.
    """

    if not lines:
        return []

    # На всякий случай гарантируем
    # порядок документа.
    lines = sorted(
        lines,
        key=lambda x: (
            x.get("page", 0),
            x["box"][1],
            x["box"][0]
        )
    )

    # Автоматически вычисляем нормальный
    # межстрочный интервал каждой страницы.
    gap_thresholds = (
        get_reference_gap_thresholds(
            lines
        )
    )

    print(
        "\n📐 Автоматические пороги "
        "абзацных интервалов:"
    )

    for page_num in sorted(
        gap_thresholds
    ):
        print(
            f"   page {page_num}: "
            f"{gap_thresholds[page_num]:.1f}"
        )

    records = []

    current = ""

    for i, line in enumerate(lines):

        text = normalize_reference_text(
            line["text"]
        )

        if not text:
            continue

        # Самая первая строка
        # всегда начинает первую запись.
        if i == 0:

            current = text
            continue

        previous_line = (
            lines[i - 1]
        )

        previous_text = (
            normalize_reference_text(
                previous_line["text"]
            )
        )

        # ==========================================
        # 1. СИЛЬНЫЙ STRUCTURAL SIGNAL
        # ==========================================

        paragraph_break = (
            has_strong_paragraph_gap(
                previous_line,
                line,
                gap_thresholds
            )
        )

        # ==========================================
        # 2. ЯВНАЯ НУМЕРАЦИЯ
        # ==========================================

        explicit_number = (
            starts_with_explicit_reference_number(
                text
            )
        )

        # ==========================================
        # 3. НЕ ДАЁМ РАЗРЕЗАТЬ СПИСОК АВТОРОВ
        # ==========================================

        author_continuation = (
            is_author_list_continuation(
                previous_text,
                text
            )
        )

        # ==========================================
        # 4. СТАРЫЙ SEMANTIC SIGNAL
        # ==========================================

        semantic_start = (
            is_new_reference(
                lines,
                i
            )
        )

        # ==========================================
        # РЕШЕНИЕ
        # ==========================================

        new_reference = False

        # Большой визуальный отступ —
        # самый сильный сигнал.
        if paragraph_break:

            new_reference = True

        # Явный номер источника —
        # тоже очень сильный сигнал.
        elif explicit_number:

            new_reference = True

        # Если визуального разрыва нет
        # и предыдущая строка закончилась
        # перечислением авторов —
        # НЕ разделяем запись,
        # даже если semantic heuristic
        # ошибочно увидела нового автора.
        elif author_continuation:

            new_reference = False

        # В остальных случаях используем
        # старую текстовую логику.
        elif semantic_start:

            new_reference = True

        # ==========================================
        # СОБИРАЕМ
        # ==========================================

        if new_reference:

            if current:

                records.append(
                    normalize_reference_text(
                        current
                    )
                )

            current = text

        else:

            current = (
                join_reference_lines(
                    current,
                    text
                )
            )

    if current:

        records.append(
            normalize_reference_text(
                current
            )
        )

    return records


def extract_references(
    all_pages_lines,
    ref_page
):
    if ref_page is None:
        return []

    reference_pages = []

    for page_num, lines in all_pages_lines.items():

        if any(
            line["label"] == "REFERENCE"
            for line in lines
        ):
            reference_pages.append(
                page_num
            )

    reference_pages.sort()

    if not reference_pages:
        return []

    selected_pages = [ref_page]

    p = ref_page - 1

    while p in reference_pages:
        selected_pages.insert(0, p)
        p -= 1

    p = ref_page + 1

    while p in reference_pages:
        selected_pages.append(p)
        p += 1

    print(
        "   Страницы библиографии: "
        + ", ".join(
            map(str, selected_pages)
        )
    )

    reference_lines = []

    for page_num in selected_pages:

        page_lines = all_pages_lines[
            page_num
        ]

        for line in page_lines:

            text = normalize_reference_text(
                line["text"]
            )

            if not text:
                continue

            # Заголовок раздела не является записью
            if is_ref_heading(text):
                continue

            # Номер страницы документа
            if is_page_number_line(line):
                continue

            item = dict(line)

            item["page"] = page_num
            item["text"] = text

            reference_lines.append(
                item
            )

    reference_lines.sort(
        key=lambda x: (
            x["page"],
            x["box"][1],
            x["box"][0]
        )
    )

    print(
        f"   🧩 Физических строк: "
        f"{len(reference_lines)}"
    )

    records = group_references(
        reference_lines
    )

    records = [
        x
        for x in records
        if len(x) >= 8
    ]

    print(
        f"   ✅ Восстановлено записей: "
        f"{len(records)}"
    )

    return records





print("\n📖 Извлекаем содержание...")
toc_lines = extract_toc(all_pages_lines, toc_page)
print(f"    Извлечено {len(toc_lines)} строк")

print("\n Извлекаем библиографию...")
ref_lines = extract_references(all_pages_lines, ref_page)
print(f"    Извлечено {len(ref_lines)} строк")


# ОЧИСТКА СОДЕРЖАНИЯ

def clean_toc(toc_items):
    """Убирает мусор из содержания."""
    cleaned = []
    for item in toc_items:
        text = item["text"].strip()

        if len(text) < 5:
            continue

        has_section = re.match(r'^\s*(\d+(\.\d+)*\.?\s+)', text) is not None
        has_page = re.search(r'\d+\s*$', text) is not None

        is_header = text.upper() in [
            "ВВЕДЕНИЕ", "ЗАКЛЮЧЕНИЕ",
            "СПИСОК ТЕРМИНОВ И АББРЕВИАТУР", "АННОТАЦИЯ"
        ]

        if has_section and has_page:
            cleaned.append(item)
        elif has_section and len(text) < 80:
            cleaned.append(item)
        elif is_header and has_page:
            cleaned.append(item)

    return cleaned


# ФОРМАТИРОВАНИЕ DOCX
def wrap_toc_title(title, max_chars=52):
    """
    Переносит длинный заголовок содержания по словам.

    Важно:
    - ничего не знает о конкретной строке;
    - работает для любого содержания;
    - не режет слова;
    - возвращает список строк.

    max_chars = эмпирический предел для строки
    при Times New Roman 14 и ваших полях.
    """

    title = re.sub(r"\s+", " ", title).strip()

    if len(title) <= max_chars:
        return [title]

    words = title.split()
    lines = []
    current = ""

    for word in words:
        candidate = word if not current else current + " " + word

        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                lines.append(current)
            current = word

    if current:
        lines.append(current)

    return lines
def render_toc(doc, toc_items):
    """
    Рисует содержание через обычные абзацы
    с правым tab stop и точками-заполнителями.

    Для длинных строк:
    - заголовок переносится вручную по словам,
    - номер страницы ставится только на последней строке,
    - все номера остаются в одном правом столбце.
    """

    if not toc_items:
        return

    toc_items = sorted(
        toc_items,
        key=lambda x: (
            x["page"],
            x["box"][1],
            x["box"][0]
        )
    )

    p = doc.add_paragraph()
    p.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
    p.paragraph_format.space_after = Pt(12)

    run = p.add_run("СОДЕРЖАНИЕ")
    run.bold = True
    run.font.name = "Times New Roman"
    run.font.size = Pt(14)

    style_name = "TOCStyle"

    if style_name in doc.styles:
        style = doc.styles[style_name]
    else:
        style = doc.styles.add_style(style_name, 1)

    style.font.name = "Times New Roman"
    style.font.size = Pt(14)

    style.paragraph_format.space_before = Pt(0)
    style.paragraph_format.space_after = Pt(0)
    style.paragraph_format.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

    style.paragraph_format.tab_stops.clear_all()
    style.paragraph_format.tab_stops.add_tab_stop(
        Cm(16.0),
        WD_TAB_ALIGNMENT.RIGHT,
        WD_TAB_LEADER.DOTS
    )

    for item in toc_items:
        text = item["text"].strip()

        if not text:
            continue

        match = re.match(r"^(.*?)(\d+)\s*$", text)

        if match:
            title = match.group(1).strip()
            page_number = match.group(2)
        else:
            title = text
            page_number = ""

        title = re.sub(r"\.{2,}\s*$", "", title)
        title = re.sub(r"\s+", " ", title).strip()

        wrapped_lines = wrap_toc_title(title, max_chars=60)

        p = doc.add_paragraph(style=style)
        p.paragraph_format.space_before = Pt(0)
        p.paragraph_format.space_after = Pt(0)
        p.paragraph_format.line_spacing = 1.5

        if len(wrapped_lines) == 1:
            p.add_run(wrapped_lines[0] + "\t" + page_number)

        else:
            for line in wrapped_lines[:-1]:
                p.add_run(line)
                p.add_run("\n")

            p.add_run(wrapped_lines[-1] + "\t" + page_number)

def render_references(doc, ref_items):
    """
    Добавляет список литературы в DOCX.
    """

    if not ref_items:
        return

    # Заголовок
    p = doc.add_paragraph()

    p.alignment = (
        WD_PARAGRAPH_ALIGNMENT.CENTER
    )

    p.paragraph_format.space_after = Pt(12)

    run = p.add_run(
        "СПИСОК ЛИТЕРАТУРЫ"
    )

    run.bold = True
    run.font.name = "Times New Roman"
    run.font.size = Pt(14)

    for i, entry in enumerate(
        ref_items,
        start=1
    ):

        entry = normalize_reference_text(
            entry
        )

        entry = re.sub(
            r"^\s*(?:"
            r"\[\d+\]"
            r"|\d+\."
            r"|\d+\)"
            r")\s*",
            "",
            entry
        )

        p = doc.add_paragraph()

        p.alignment = (
            WD_PARAGRAPH_ALIGNMENT.JUSTIFY
        )

        p.paragraph_format.first_line_indent = Cm(0)

        p.paragraph_format.left_indent = Cm(0)

        p.paragraph_format.space_before = Pt(0)

        p.paragraph_format.space_after = Pt(6)

        run = p.add_run(
            f"{i}. {entry}"
        )

        run.font.name = "Times New Roman"
        run.font.size = Pt(14)

# СОХРАНЕНИЕ

doc = Document()
section = doc.sections[0]
section.top_margin = Cm(2)
section.bottom_margin = Cm(2)
section.left_margin = Cm(3)
section.right_margin = Cm(1.5)

render_toc(doc, toc_lines)
if toc_lines and ref_lines:
    doc.add_page_break()
render_references(doc, ref_lines)

doc.save(OUTPUT_DOCX)
print(f"\n Файл сохранён: {OUTPUT_DOCX}")
print(f" Итог: TOC строк {len(toc_lines)}, REFERENCE строк {len(ref_lines)}")

📄 Извлекаем текст из PDF через PyMuPDF...
   Извлечено 39 страниц
 Классифицируем страницы.
   ✅ Обработано 39 страниц

🔍 Ищем страницы по плотности меток...
    Страница 2: 20 строк TOC, плотность 0.91
    Страница 37: 28 строк REFERENCE, плотность 0.97
    Страница содержания: 2
    Страница библиографии: 37

📖 Извлекаем содержание...
   📑 Страницы содержания: 2
    Извлечено 20 строк

 Извлекаем библиографию...
   Страницы библиографии: 37, 38, 39
   🧩 Физических строк: 72

📐 Автоматические пороги абзацных интервалов:
   page 37: 11.0
   page 38: 11.0
   page 39: 11.0
   ✅ Восстановлено записей: 23
    Извлечено 23 строк

 Файл сохранён: /content/drive/MyDrive/GOST_Project/smart_result_new2.docx
 Итог: TOC строк 20, REFERENCE строк 23
